# Identity Resolution (Advanced, Unique)

## Beginner-friendly summary
This notebook links fragmented records that may belong to the same real person and shows how identity quality affects downstream analytics.

### What this notebook does
- Creates/loads identity-style records across devices/channels
- Builds similarity features between candidate pairs
- Trains a linkage classifier for match vs non-match
- Forms resolved entities/groups from predicted links
- Evaluates match quality and coverage
- Analyzes effects of missing signals on match rate
- Demonstrates cross-channel stitching outputs

### Major steps (in order)
1. Setup and synthetic/real-like data preparation
2. Candidate pair construction
3. Similarity feature engineering
4. Linkage model training and scoring
5. Entity resolution and grouping
6. Quality checks and diagnostics
7. Match-rate sensitivity analysis
8. Cross-channel stitching summary

### Side notes for beginners
- Record linkage asks: “Do these two rows represent the same person?”
- Entity resolution combines many pairwise links into one resolved identity graph/group.
- Match rate can drop quickly when key identifiers are missing or noisy.


## 1. Setup

> Side note: this section imports libraries and sets reproducible seeds.

In [ ]:
# Side note (beginner): import only what we need for synthetic generation,
# similarity features, and supervised linkage modeling.

from __future__ import annotations

import random
import hashlib
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

np.random.seed(42)
random.seed(42)

## 2. Build Synthetic Identity Data

We create:
- a hidden `person_id` (ground truth identity)
- multiple observed records per person (`record_id`) across devices
- noisy/missing attributes (email variants, phone, IP, user agent, cookie)

> Side note: in production, we never observe true `person_id`; here it is used only for evaluation.

In [ ]:
@dataclass
class Person:
    person_id: str
    first_name: str
    last_name: str
    email: str
    phone: str
    city: str


first_names = [
    'alex', 'sam', 'jamie', 'taylor', 'casey', 'jordan', 'morgan', 'riley', 'chris', 'devon',
    'jess', 'kiran', 'deepa', 'maria', 'liam', 'olivia', 'noah', 'ava', 'mia', 'zoe'
]
last_names = [
    'smith', 'johnson', 'lee', 'patel', 'garcia', 'williams', 'brown', 'miller', 'wilson', 'moore'
]
domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'company.com']
cities = ['sf', 'nyc', 'la', 'chicago', 'austin', 'seattle']
user_agents = ['ios_safari', 'android_chrome', 'mac_chrome', 'windows_edge', 'ipad_safari']


def mk_phone() -> str:
    return ''.join(str(np.random.randint(0, 10)) for _ in range(10))


def email_variant(email: str) -> str:
    # mimic plus aliases / punctuation / occasional typo
    local, domain = email.split('@')
    v = local
    if np.random.rand() < 0.35:
        v = v + '+promo'
    if np.random.rand() < 0.20:
        v = v.replace('.', '')
    if np.random.rand() < 0.08 and len(v) > 3:
        i = np.random.randint(0, len(v))
        v = v[:i] + chr(ord('a') + np.random.randint(0, 26)) + v[i+1:]
    return f'{v}@{domain}'


def maybe_missing(value: str, p: float = 0.12) -> str:
    return value if np.random.rand() > p else ''


def hash_email(email: str) -> str:
    if not email:
        return ''
    return hashlib.sha256(email.encode()).hexdigest()[:16]


n_people = 450
people: List[Person] = []
for i in range(n_people):
    fn = random.choice(first_names)
    ln = random.choice(last_names)
    local = f'{fn}.{ln}{np.random.randint(1,999)}'
    email = f'{local}@{random.choice(domains)}'
    person = Person(
        person_id=f'P{i:04d}',
        first_name=fn,
        last_name=ln,
        email=email,
        phone=mk_phone(),
        city=random.choice(cities),
    )
    people.append(person)


records: List[Dict] = []
record_ctr = 0
for p in people:
    n_records = np.random.randint(2, 6)  # fragmented observations
    for _ in range(n_records):
        em = email_variant(p.email)
        em = maybe_missing(em, p=0.10)
        ph = maybe_missing(p.phone, p=0.15)
        city = maybe_missing(p.city, p=0.08)
        ua = random.choice(user_agents)
        ip_prefix = f'10.{np.random.randint(0,256)}.{np.random.randint(0,256)}'
        ip = f'{ip_prefix}.{np.random.randint(1,255)}'
        cookie = hashlib.md5(f'{p.person_id}_{ua}_{np.random.randint(0,99999)}'.encode()).hexdigest()[:12]

        records.append({
            'record_id': f'R{record_ctr:06d}',
            'person_id': p.person_id,
            'name': f'{p.first_name} {p.last_name}',
            'email': em,
            'email_hash': hash_email(em),
            'phone': ph,
            'city': city,
            'user_agent': ua,
            'ip': ip,
            'cookie_id': cookie,
        })
        record_ctr += 1

df = pd.DataFrame(records)
print({'n_people': n_people, 'n_records': len(df), 'avg_records_per_person': round(len(df)/n_people, 2)})
df.head(5)

## 3. Candidate Pair Generation (Blocking)

Record linkage normally compares only plausible pairs (blocking), not all combinations.

Blocking keys used here:
- same email domain
- OR same phone last-4 digits
- OR same city + same user agent

> Side note: blocking is key for scalability because naive all-pairs grows as O(N²).

In [ ]:
# Side note (beginner): create likely candidate pairs, then label them with
# hidden truth (`same_person`) only for training/evaluation.
# We also cap pair volume to keep notebook fast.

MAX_BLOCK_SIZE = 120
MAX_CANDIDATE_PAIRS = 70000

def email_domain(e: str) -> str:
    return e.split('@')[1] if isinstance(e, str) and '@' in e else ''


def phone_last4(p: str) -> str:
    return p[-4:] if isinstance(p, str) and len(p) >= 4 else ''


tmp = df.copy()
tmp['email_domain'] = tmp['email'].map(email_domain)
tmp['phone_last4'] = tmp['phone'].map(phone_last4)
tmp['city_ua'] = tmp['city'].fillna('') + '|' + tmp['user_agent'].fillna('')

idx_by_domain = tmp.groupby('email_domain').groups
idx_by_phone4 = tmp.groupby('phone_last4').groups
idx_by_city_ua = tmp.groupby('city_ua').groups

pair_set = set()

def add_pairs(indexes):
    idx = list(indexes)
    if len(idx) < 2:
        return
    if len(idx) > MAX_BLOCK_SIZE:
        idx = random.sample(idx, MAX_BLOCK_SIZE)
    for a_pos in range(len(idx)):
        for b_pos in range(a_pos + 1, len(idx)):
            a, b = idx[a_pos], idx[b_pos]
            pair_set.add((a, b) if a < b else (b, a))
            if len(pair_set) >= MAX_CANDIDATE_PAIRS:
                return


for k, g in idx_by_domain.items():
    if k and len(pair_set) < MAX_CANDIDATE_PAIRS:
        add_pairs(g)
for k, g in idx_by_phone4.items():
    if k and len(pair_set) < MAX_CANDIDATE_PAIRS:
        add_pairs(g)
for k, g in idx_by_city_ua.items():
    if k != '|' and len(pair_set) < MAX_CANDIDATE_PAIRS:
        add_pairs(g)

pairs = pd.DataFrame(list(pair_set), columns=['i', 'j'])
pairs['same_person'] = (tmp.loc[pairs['i'], 'person_id'].values == tmp.loc[pairs['j'], 'person_id'].values).astype(int)

print({'candidate_pairs': int(len(pairs)), 'positive_rate': round(float(pairs['same_person'].mean()), 4)})
pairs.head(5)

## 4. Similarity Features (Cosine + Deterministic Matches)

Feature families:
- embedding cosine similarity from profile text
- exact/partial key matches (email hash, domain, phone last4, city, UA)
- token overlap on names

> Side note: in identity systems, we typically combine deterministic keys and probabilistic similarity.

In [ ]:
# Side note (beginner): build numeric pair-features that a classifier can learn from.

def profile_text(row: pd.Series) -> str:
    vals = [
        str(row.get('name', '')).lower(),
        str(row.get('email', '')).lower(),
        str(row.get('phone', '')),
        str(row.get('city', '')).lower(),
        str(row.get('user_agent', '')).lower(),
        str(row.get('ip', '')).lower(),
    ]
    return ' '.join(v for v in vals if v and v != 'nan')


txt = tmp.apply(profile_text, axis=1)
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2)
X_tfidf = vectorizer.fit_transform(txt)

# dense embeddings (compressed text vectors)
svd = TruncatedSVD(n_components=48, random_state=42)
E = svd.fit_transform(X_tfidf)


def name_token_jaccard(a: str, b: str) -> float:
    sa = set(str(a).lower().split())
    sb = set(str(b).lower().split())
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


feat_rows = []
for _, r in pairs.iterrows():
    i, j = int(r['i']), int(r['j'])
    ri = tmp.loc[i]
    rj = tmp.loc[j]

    cos_sim = float(cosine_similarity(E[i].reshape(1, -1), E[j].reshape(1, -1))[0, 0])

    feat_rows.append({
        'i': i,
        'j': j,
        'same_person': int(r['same_person']),
        'cosine_profile_emb': cos_sim,
        'email_hash_match': int(ri['email_hash'] != '' and ri['email_hash'] == rj['email_hash']),
        'email_domain_match': int(email_domain(ri['email']) != '' and email_domain(ri['email']) == email_domain(rj['email'])),
        'phone_last4_match': int(phone_last4(ri['phone']) != '' and phone_last4(ri['phone']) == phone_last4(rj['phone'])),
        'city_match': int(str(ri['city']) != '' and str(ri['city']) == str(rj['city'])),
        'ua_match': int(str(ri['user_agent']) == str(rj['user_agent'])),
        'name_jaccard': name_token_jaccard(ri['name'], rj['name']),
    })

feat_df = pd.DataFrame(feat_rows)
feature_cols = [c for c in feat_df.columns if c not in ['i', 'j', 'same_person']]
feat_df.head(5)

## 5. Linkage Classification Model

We train a pairwise classifier:
- input: pair similarity features
- output: probability that two records belong to same person

> Side note: this is a classic supervised record-linkage design.
>
> Beginner note: Logistic Regression here is a **pair scorer**. It answers:
> "Do record A and record B look like the same person?" It does not build final clusters by itself.

In [ ]:
# Side note (beginner): train/test evaluate how well model predicts true matches.

X = feat_df[feature_cols].values
y = feat_df['same_person'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)

p_test = clf.predict_proba(X_test)[:, 1]
pred = (p_test >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, p_test)
pr_auc = average_precision_score(y_test, p_test)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)

print({
    'roc_auc': round(float(roc_auc), 4),
    'pr_auc': round(float(pr_auc), 4),
    'precision@0.5': round(float(prec), 4),
    'recall@0.5': round(float(rec), 4),
    'f1@0.5': round(float(f1), 4),
})

## 6. Resolve Identities (Graph Linkage)

Now we convert pair probabilities into resolved IDs:
- keep high-confidence links
- connect links as graph components
- assign one `resolved_id` per connected component

> Side note: this is how pairwise matching becomes identity graph stitching.
>
> Beginner note: Graph Linkage is the **stitching step** after pair scoring.
> Think of each record as a dot and each high-confidence match as a line.
> All connected dots become one resolved identity.

In [ ]:
# Side note (beginner): use union-find to merge linked records into clusters.
# Pair model says which links are likely; union-find stitches all linked records together.

pair_probs = clf.predict_proba(feat_df[feature_cols].values)[:, 1]
link_threshold = 0.70
linked_pairs = feat_df.loc[pair_probs >= link_threshold, ['i', 'j']].copy()

parent = {idx: idx for idx in tmp.index}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for _, row in linked_pairs.iterrows():
    union(int(row['i']), int(row['j']))

tmp['resolved_id'] = [find(i) for i in tmp.index]

# cluster quality proxy: each resolved cluster should mostly map to one person_id
cluster_purity = (
    tmp.groupby('resolved_id')['person_id']
       .apply(lambda s: s.value_counts(normalize=True).iloc[0])
       .mean()
)

print({
    'n_resolved_clusters': int(tmp['resolved_id'].nunique()),
    'linked_pairs': int(len(linked_pairs)),
    'avg_cluster_purity': round(float(cluster_purity), 4)
})

# Side note (beginner): explicitly display resolved rows so output is always visible.
from IPython.display import display

resolved_view = tmp[['record_id', 'person_id', 'resolved_id', 'email', 'phone', 'city', 'user_agent']].head(20)
print('\nSample resolved entities (first 20 rows):')
display(resolved_view)

cluster_sizes = (
    tmp.groupby('resolved_id')
       .size()
       .reset_index(name='n_records')
       .sort_values('n_records', ascending=False)
       .head(10)
)
print('\nTop resolved clusters by size:')
display(cluster_sizes)

# Side note (beginner): this quality table is only available in synthetic data,
# because it needs hidden ground-truth `person_id`.
print('\nQuality check (synthetic data): unique true persons per resolved cluster')
if 'person_id' not in tmp.columns:
    print('`person_id` not found. Skipping ground-truth quality check.')
else:
    quality = (
        tmp.groupby('resolved_id')['person_id']
           .nunique()
           .reset_index(name='unique_true_person_ids')
           .sort_values('unique_true_person_ids', ascending=False)
    )
    display(quality.head(20))

## 6b. Signal Loss / Data Quality Impact

Client question: **Why is measurement broken?**

Beginner side notes:
- Privacy changes can remove key signals (email hash, device IDs, geo hints).
- We simulate missing-signal scenarios by removing feature groups.
- Then we compare model quality drop (ROC-AUC / PR-AUC).
- Bigger drop means stronger business dependence on that signal group.


In [ ]:
# Side note (beginner): positive performance drop = quality degradation when signal is removed.

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_full = feat_df[feature_cols].values
y_full = feat_df['same_person'].values

Xtr, Xte, ytr, yte = train_test_split(
    X_full, y_full, test_size=0.25, random_state=42, stratify=y_full
)

base_model = LogisticRegression(max_iter=1000, class_weight='balanced')
base_model.fit(Xtr, ytr)
p_base = base_model.predict_proba(Xte)[:, 1]
base_roc = float(roc_auc_score(yte, p_base))
base_pr = float(average_precision_score(yte, p_base))

groups = {
    'all_signals': feature_cols,
    'drop_email_signals': [c for c in feature_cols if c not in ['email_hash_match', 'email_domain_match']],
    'drop_device_signals': [c for c in feature_cols if c not in ['ua_match', 'phone_last4_match']],
    'drop_geo_signals': [c for c in feature_cols if c not in ['city_match']],
    'drop_identity_keys': [c for c in feature_cols if c not in ['email_hash_match', 'email_domain_match', 'phone_last4_match', 'city_match', 'ua_match']],
}

rows = []
for name, cols in groups.items():
    Xt = feat_df[cols].values
    Xtr_g, Xte_g, ytr_g, yte_g = train_test_split(
        Xt, y_full, test_size=0.25, random_state=42, stratify=y_full
    )
    m = LogisticRegression(max_iter=1000, class_weight='balanced')
    m.fit(Xtr_g, ytr_g)
    p = m.predict_proba(Xte_g)[:, 1]
    roc = float(roc_auc_score(yte_g, p))
    pr = float(average_precision_score(yte_g, p))
    rows.append({
        'scenario': name,
        'roc_auc': roc,
        'pr_auc': pr,
        'roc_drop_vs_full': base_roc - roc,
        'pr_drop_vs_full': base_pr - pr,
    })

signal_loss_report = pd.DataFrame(rows).sort_values('pr_drop_vs_full', ascending=False)
display(signal_loss_report)


## 6c. Match Rate Optimization (Feature Completeness)

Client question: **Why is match rate low?**

Beginner side notes:
- If core fields are missing, linkage confidence drops.
- We measure pair completeness and compare predicted/true match rates by bucket.
- This gives a data-quality action plan (which fields to fix first).


In [ ]:
# Side note (beginner): completeness here is based on whether key attributes are present.

pair_probs_all = clf.predict_proba(feat_df[feature_cols].values)[:, 1]
pair_df = feat_df[['i', 'j', 'same_person']].copy()
pair_df['pred_match'] = (pair_probs_all >= 0.70).astype(int)

key_fields = ['email_hash', 'phone', 'city', 'user_agent', 'cookie_id']
record_completeness = tmp[key_fields].replace('', np.nan).notna().mean(axis=1)
pair_df['pair_completeness'] = (record_completeness.loc[pair_df['i']].values + record_completeness.loc[pair_df['j']].values) / 2.0

pair_df['completeness_bucket'] = pd.cut(
    pair_df['pair_completeness'],
    bins=[0, 0.4, 0.6, 0.8, 1.0],
    include_lowest=True,
    labels=['0-40%', '40-60%', '60-80%', '80-100%']
)

match_rate_report = (
    pair_df.groupby('completeness_bucket', observed=False)
           .agg(
               n_pairs=('same_person', 'size'),
               predicted_match_rate=('pred_match', 'mean'),
               true_match_rate=('same_person', 'mean')
           )
           .reset_index()
)
display(match_rate_report)


## 6d. Cross-Channel User Stitching (Journey Paths)

Client question: **How users move across channels?**

Beginner side notes:
- After identity resolution, we can aggregate events at resolved-user level.
- We simulate multi-touch events per record, stitch by `resolved_id`, and build path strings.
- Output: most common conversion journeys across channels.


In [ ]:
# Side note (beginner): this creates an event table and then stitches paths by resolved identity.

rng = np.random.default_rng(42)
channels = np.array(['search', 'social', 'display', 'email', 'direct'])

events = []
for ridx, row in tmp.iterrows():
    n_events = int(rng.integers(2, 6))
    ch = rng.choice(channels, size=n_events, replace=True)
    conv = int(rng.random() < 0.22)
    for i in range(n_events):
        events.append({
            'record_id': row['record_id'],
            'resolved_id': row['resolved_id'],
            'event_order': i + 1,
            'channel': ch[i],
            'conversion_event': int(conv and i == n_events - 1),
        })

events_df = pd.DataFrame(events)

journey_df = (
    events_df.sort_values(['resolved_id', 'event_order'])
             .groupby('resolved_id')
             .agg(
                 path=('channel', lambda s: ' > '.join(s.tolist())),
                 converted=('conversion_event', 'max')
             )
             .reset_index()
)

top_paths = (
    journey_df[journey_df['converted'] == 1]
    .groupby('path')
    .size()
    .reset_index(name='n_converted_users')
    .sort_values('n_converted_users', ascending=False)
    .head(15)
)

print('Sample stitched journeys:')
display(journey_df.head(10))
print('Top conversion paths:')
display(top_paths)


## Column Lineage (Where Each Field Comes From)

- `record_id`: generated per observed record (`R000001`, `R000002`, ...).
- `person_id`: hidden ground-truth person key used only for synthetic evaluation.
- `resolved_id`: model output cluster ID from graph linkage (union-find over linked pairs).
- `unique_true_person_ids`: quality-check metric computed as distinct `person_id` count inside each `resolved_id` cluster.

> Side note: in real production identity resolution, `person_id` is not known, so `unique_true_person_ids` is unavailable.

## 7. Key Learning

- Identity resolution connects fragmented records across devices into one profile.
- Deterministic keys and cosine-based embedding similarity complement each other.
- Pairwise classification + graph linkage is a practical production pattern.

> Core takeaway: connect fragmented user data into one identity.

---
## Recommended Learning Order
1. Uplift modeling
2. Attribution / MMM
3. CTR + calibration
4. Identity resolution